<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="../images/nvidia.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building LLM Applications With Prompt Engineering</b></font></h1>
<h2><b>Reasoning Models</b></h2>
<br>


## License

Unless otherwise noted:

- Code cells are licensed under the
  [Apache License 2.0](../../../LICENSES/Apache-2.0.txt).
- Markdown and raw cells are licensed under the
  [Creative Commons Attribution 4.0 International License](../../../LICENSES/CC-BY-4.0.txt).
- Third-party material retains its original license and is identified in
  [`THIRD_PARTY_NOTICES.md`](../../../THIRD_PARTY_NOTICES.md).

Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES.

This notebook compares two configurations of `nvidia/nemotron-3.5-lightning-30b-a3b`: direct responses with thinking disabled, and explicit reasoning with a bounded budget. Both use the same sampling settings.

The endpoint keeps the final answer in `.content` and may expose a reasoning trace in `additional_kwargs['reasoning_content']`. Compare answer quality and inspect the trace before choosing a configuration.

In the previous notebook, we learned about Chain-of-Thought (CoT) prompting, a technique where we ask a general chat model to work step by step. What changes when a model also provides a reasoning mode designed for that work?

In this notebook, we'll request that mode explicitly, inspect its separate reasoning metadata, and compare its trade-offs with a direct response.


```mermaid
flowchart TB
    subgraph N["Non-reasoning call"]
      N1["Prompt"]:::box --> N2["Model"]:::box --> N3["Visible answer text"]:::answer
    end

    subgraph R["Reasoning call"]
      R1["Prompt"]:::box --> R2["Reasoning mode"]:::gpu
      R2 --> R3["Visible answer text"]:::answer
      R2 -.-> R4["Reasoning metadata<br>reasoning_content"]:::hidden
    end

    classDef box fill:#161b22,stroke:#30363d,stroke-width:1.5px,color:#e6edf3
    classDef gpu fill:#211b12,stroke:#f2cc60,stroke-width:2px,color:#f2cc60
    classDef answer fill:#0d2b40,stroke:#58a6ff,stroke-width:2px,color:#58a6ff
    classDef hidden fill:#1f1a2e,stroke:#8b5cf6,stroke-width:2px,color:#c4b5fd,stroke-dasharray:5 5
```

---


## Objectives


By the time you complete this notebook you will:

- Distinguish the direct-response and reasoning model roles
- Recognize the latency and token trade-offs of the reasoning path
- Read reasoning metadata returned separately by the endpoint
- Integrate the reasoning model into a LangChain chain

---


## Imports


In [ ]:
import os
import re
import time
import warnings

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage

warnings.filterwarnings(
    "ignore",
    message=r"^Reasoning content was parsed from <think> tags in model output\.",
    category=UserWarning,
    module=r"^langchain_nvidia_ai_endpoints\.chat_models$",
)

warnings.filterwarnings(
    "ignore",
    message=r"^Found nvidia/nemotron-3\.5-lightning-30b-a3b in available_models, but type is unknown and inference may fail\.",
    category=UserWarning,
    module=r"^langchain_nvidia_ai_endpoints\._common$",
)


---


## Create Model Instances


We'll compare the course's direct-response model with a model intended for explicit reasoning.

The direct path uses [**`nvidia/nemotron-3.5-lightning-30b-a3b`**](https://build.nvidia.com/nvidia/nemotron-3.5-lightning-30b-a3b/modelcard). The reasoning path uses [**`nvidia/nemotron-3.5-lightning-30b-a3b`**](https://build.nvidia.com/nvidia/nemotron-3.5-lightning-30b-a3b/modelcard) with `enable_thinking` and a bounded reasoning budget. Both use the same OpenAI-compatible interface.

### API key for this notebook kernel

Paste the API key for your configured endpoint when the next cell prompts for it. The typed value is visible during entry but is not placed in the cell source or written to a file. It remains only in this kernel's memory. Restarting the kernel discards it, and a separate notebook kernel will ask for the key again.

Avoid entering the key while screen sharing or recording.

In [ ]:
import os

if not os.environ.get("NVIDIA_API_KEY"):
    _api_key = input("Paste the API key for the configured endpoint: ").strip()
    if not _api_key:
        raise ValueError("An API key is required to run this notebook.")
    os.environ["NVIDIA_API_KEY"] = _api_key
    del _api_key

os.environ.setdefault("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")

# Alternative endpoint example (Nebius Token Factory):
# os.environ["NVIDIA_BASE_URL"] = "https://api.tokenfactory.nebius.com/v1/"
print("API access is configured for this notebook kernel.")

In [ ]:
base_url = os.getenv("NVIDIA_BASE_URL")
model = 'nvidia/nemotron-3.5-lightning-30b-a3b'
# Alternative model ID for Nebius Token Factory:
# model = 'nvidia/Nemotron-3_5-Lightning'

# Hold the model and sampling settings fixed; vary the thinking configuration.
standard_llm = ChatNVIDIA(
    base_url=base_url, model=model, temperature=1, top_p=0.95, max_completion_tokens=8192,
    model_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
)

# Enable a bounded reasoning trace for the comparison.
reasoning_llm = ChatNVIDIA(
    base_url=base_url, model=model, temperature=1, top_p=0.95, max_completion_tokens=8192,
    model_kwargs={"chat_template_kwargs": {"enable_thinking": True}, "reasoning_budget": 1024},
)

---


## Streaming Printing Helper

In this notebook we will use the following helper function to print streaming responses from the LLM.

In [ ]:
def sprint(stream):
    for chunk in stream:
        if chunk.additional_kwargs.get('reasoning_content'):    ## Display reasoning metadata when present.
            print(chunk.additional_kwargs.get('reasoning_content', ""), end='', flush=True)
        if chunk.content:
            print("\033[1m" + chunk.content + "\033[0m", end='', flush=True) ## Otherwise, we will print output in bold.

---

## Reasoning Models

In the previous notebook, a Chain-of-Thought prompt asked the direct-response model to show intermediate steps. Here, the reasoning model requests a separate reasoning trace through its serving interface.

We will give both models the same multi-step logic puzzle, compare their final answers first, then inspect the reasoning response shape.

In [ ]:
problem = """You are a world-class logician. Read the following short story carefully:
Alice, Bob, and Charlie are the only three people in a house. At midnight, exactly one of them is in the kitchen, exactly one is in the bedroom, and exactly one is in the bathroom.
The light in the kitchen is on.
Alice hates bright lights and only stays in rooms while their lights are off.
Bob is afraid of the dark and only stays in rooms while their lights are on.
Charlie always tells the truth.
Charlie says: “Bob is in the bathroom.”
Where is each person, and is the bedroom light on or off?
Answer with exactly four sentences: one stating where Alice is, one stating where Bob is, one stating where Charlie is, and one stating whether the bedroom light is on or off.
Do not guess."""

If you're interested try to work the problem out for yourself. When you're ready view the solution by viewing the cell immediately below.

<details><summary><b>Click to show solution</b></summary>
<blockquote><pre>Alice is in the bedroom.  
Bob is in the bathroom.  
Charlie is in the kitchen.  
The bedroom light is off.
</pre></blockquote></details>

First, let's see how Nemotron 3.5 Lightning handles the logic puzzle.

In [ ]:
## Diagnostic Check: What's actually coming out of the stream.
# stream = standard_llm.stream(problem)
# print(next(stream))
# print(next(stream))
sprint(standard_llm.stream(problem))

Now give the same problem to Nemotron 3.5 Lightning's reasoning path.

In [ ]:
## Diagnostic Check: What's actually coming out of the stream.
# stream = reasoning_llm.stream([("user", problem)])
# print(next(stream))
# print(next(stream))

sprint(reasoning_llm.stream([("user", problem)]))

Compare both final answers with the revealed solution before comparing their response shape. A longer answer or reasoning trace is not evidence that the answer is correct. The direct call returns answer content, while the reasoning-enabled call can also stream a trace through `reasoning_content`.

The serving API separates that trace from the final answer, so LangChain exposes it in the message's `additional_kwargs` rather than ordinary message content.


---


## The Trade-Offs


Reasoning can improve an answer while using more output tokens. Compare the delivered responses on a simple task below. Elapsed time includes transport and possible replay; it does not by itself measure inference cost.

In [ ]:
simple_task = "Summarize the benefits of exercise in one sentence."

# Time the direct-response configuration
start = time.time()
standard_response = standard_llm.invoke(simple_task)
standard_time = time.time() - start

# Time the reasoning-enabled configuration
start = time.time()
reasoning_response = reasoning_llm.invoke([("user", simple_task)])
reasoning_time = time.time() - start

standard_text = standard_response.content or ""
reasoning_text = reasoning_response.content or ""
reasoning_trace = reasoning_response.additional_kwargs.get("reasoning_content", "")

print(f"Direct response: {standard_time:.1f} seconds, {len(standard_text)} answer characters")
print(f"Reasoning mode:  {reasoning_time:.1f} seconds, {len(reasoning_text)} answer characters")
print(f"Reasoning trace: {len(reasoning_trace)} separate metadata characters")
print(f"\nReasoning/direct latency ratio: {reasoning_time/standard_time:.1f}x")


Do the answers differ in quality or useful detail? Inspect token usage and any reasoning trace. Use an explicit fresh request when measuring provider inference, and compare several runs before drawing a latency conclusion.

---


## Handling Reasoning Traces


As we saw above, our NIM-hosted model keeps streaming reasoning in `additional_kwargs['reasoning_content']`. For a completed non-streaming response, the same field is available through `additional_kwargs['reasoning_content']`, while `.content` carries the final answer. Not all reasoning models or endpoints work this way. There are two common paradigms you'll encounter:

1. **Structured API fields** — The serving infrastructure parses thinking server-side and places it in a separate response field (like `reasoning_content`). This is what our NIM endpoint does.
2. **Inline `<think>` tags** — Some models or self-hosted endpoints leave `<think>...</think>` tags directly in the response text.

To build robust applications, it's good practice to handle both cases. Let's create a helper function that does this.

In [ ]:
test_message = "<think>Let me think about this for a second...</think>Thinking is great!"
re.sub(r'<think>.*?</think>', '', test_message, flags=re.DOTALL)

Let's create a simple helper function using the same method to work on model responses.

In [ ]:
def strip_thinking(message):
    """Return the final answer while dropping model-specific reasoning traces.

    Some reasoning models emit <think>...</think> in visible content.
    Nemotron 3.5 Lightning exposes reasoning in message metadata instead, while
    message.content carries the final answer.
    """
    content = message.content or ""
    cleaned = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL)
    if '<think>' in cleaned:
        cleaned = re.sub(r'<think>.*', '', cleaned, flags=re.DOTALL)
    return AIMessage(content=cleaned.strip(), additional_kwargs={})


Let's test this helper with a fresh response from our reasoning model. Since our NIM endpoint places thinking in `additional_kwargs`, the `.content` field already contains only the final answer. The function returns that content; for a response with inline `<think>` tags, it removes the tagged block.

In [ ]:
response = reasoning_llm.invoke([("user", "What is the capital of France?")])

In [ ]:
print(response.content)
print("\nReasoning trace preview:")
print(response.additional_kwargs.get("reasoning_content", "")[:500])


In [ ]:
stripped_response = strip_thinking(response)

In [ ]:
print(stripped_response.content)

In [ ]:
visible_content = response.content or ""
reasoning_trace = response.additional_kwargs.get("reasoning_content", "")
print(f"Visible answer before stripping: {len(visible_content)} characters")
print(f"Reasoning trace side channel: {len(reasoning_trace)} characters")
print(f"After stripping: {len(stripped_response.content)} characters")


**Important Note:** The format of reasoning traces is not standardized across models. Some models use visible `<think>` tags; this endpoint exposes reasoning in `additional_kwargs` under `reasoning_content` while keeping the final answer in `content`. Always check the model documentation and inspect one response before you wire a reasoning model into a parser or tool call.

That said, the principle remains the same: you'll need to handle this content appropriately for your use case.


---


## Integrating into Chains


Now, how do we use our `strip_thinking` function in a LangChain chain? If you recall from earlier notebooks, we can use `RunnableLambda` to wrap any Python function and make it part of a chain.

Let's build a chain that uses the reasoning model and cleanly strips out the thinking process.


In [ ]:
reasoning_prompt = ChatPromptTemplate.from_messages([
    ("user", "{input}"),
])

clean_reasoning_chain = reasoning_prompt | reasoning_llm | RunnableLambda(strip_thinking) | StrOutputParser()

In [ ]:
result = clean_reasoning_chain.invoke({"input": "In one sentence, what's the best thing anyone of any age can do?"})
print(result)

The chain works as follows:
1. `reasoning_llm` generates a response with a final answer plus model-specific reasoning metadata.
2. `RunnableLambda(strip_thinking)` keeps the final answer and drops visible or side-channel reasoning traces.
3. `StrOutputParser()` extracts the clean string content.

The user or downstream runnables in the chain see only the polished final answer, while the model still benefits from its internal reasoning process.


---


## Why Stripping Matters: Structured Output


You might wonder: "Why remove a visible `<think>` block? Can't I just ignore it?"

Our current endpoint keeps the reasoning trace out of `.content`, so its trace does not break the JSON parser. Other endpoints may return inline `<think>` tags, and those tags do break a JSON-only contract. The following message reproduces that response shape before we use the live endpoint.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "Respond with JSON only, no additional text. Extract name and age as JSON: {text}")
])

parser = JsonOutputParser() # Will parse key entities in the final output into a JSON object

First, let's try **without** removing the inline reasoning block.


In [ ]:
inline_response = AIMessage(content='<think>Extract the fields.</think>{"name": "John Smith", "age": 35}')

try:
    result = parser.invoke(inline_response)
    print(f"Result: {result}")
except Exception as e:
    print(f"Error: {type(e).__name__}")
    print("The <think> block broke the JSON parser!")

Now let's try **with** our `strip_thinking` function in the chain.


In [ ]:
chain_with_strip = prompt | reasoning_llm | RunnableLambda(strip_thinking) | parser

result = chain_with_strip.invoke({"text": "John Smith is 35 years old."})
print(f"Result: {result}")

The synthetic inline response fails until the thinking block is removed. The live chain also succeeds because `strip_thinking` accepts either response shape before the JSON parser.


---


## When to Retain a Reasoning Trace


Most applications should keep reasoning traces out of the user-facing answer. Retaining them can still help in bounded cases such as:

- **Interface demonstrations**: Showing how answer content and response metadata differ
- **Endpoint diagnostics**: Checking whether a serving change moved or reformatted the trace

Treat a reasoning trace as model-generated output, not a faithful explanation or a reason to trust the answer. If you display one, label that limitation and keep it separate from the final answer.


---


## Exercise: Build a Reasoning Chain


For this exercise, you'll build a chain that uses the reasoning model for a math problem, strips any visible `<think>` block, and returns just the clean answer.

Here's the problem:

In [ ]:
challenging_expression = "286899 * 3902789"

Let's see how our `standard_llm` does with this problem.

In [ ]:
# print(standard_llm.invoke(f"What is {challenging_expression}?").content)
sprint(standard_llm.stream(f"What is {challenging_expression}?"))
print(f"ACTUAL ANSWER: {challenging_expression} = {eval(challenging_expression):,}")

Your task:
1. Create a chain using `reasoning_llm`
2. Include `strip_thinking` in the chain to remove any inline reasoning block
3. Use `StrOutputParser` to get clean string output
4. Invoke the chain with the provided problem

Feel free to check out the *Solution* below if you get stuck.

### Your Work Here


In [ ]:
# Your code here

### Solution


In [ ]:
exercise_prompt = ChatPromptTemplate.from_messages([
    ("user", "What is {expression}?"),
])
exercise_chain = exercise_prompt | reasoning_llm | RunnableLambda(strip_thinking) | StrOutputParser()

print(exercise_chain.invoke({"expression": challenging_expression}))
print(f"ACTUAL ANSWER: {challenging_expression} = {eval(challenging_expression):,}")

---


## Summary


In this notebook, you learned:

- Nemotron 3.5 Lightning serves the course's direct-response work
- Nemotron 3.5 Lightning serves the explicit reasoning path with a bounded budget
- Reasoning may appear in inline `<think>...</think>` tags or a separate `reasoning_content` field
- Quality, latency, and token use should determine whether a task uses the reasoning path
- `RunnableLambda` can keep reasoning traces away from downstream parsers

---

<!-- NEXT_STEP_CARD -->

<div style="border-left: 6px solid #76B900; background: #f7fdf2; padding: 14px 18px; border-radius: 10px; margin: 20px 0;">
<p style="margin: 0 0 6px; color: #315c00; font-weight: 700; letter-spacing: .04em; text-transform: uppercase;">Continue the unified course path</p>
<p style="margin: 0 0 8px; color: black;"><strong>Next step:</strong> Open <code>3-Prompting-With-Messages/36-Chatbots.ipynb</code> next: <strong>Chatbots</strong>.</p>
<p style="margin: 0; color: black;"><strong>Before moving on:</strong> Keep one judgment about when requested reasoning is worth its cost, because the next notebook folds those message choices into a longer conversational assistant.</p>
</div>